## 1. 라이브러리 로드

In [1]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [2]:
df_platinum_Match = pd.read_csv('../../tft_game_dataset/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('../../tft_game_dataset/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('../../tft_game_dataset/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('../../tft_game_dataset/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('../../tft_game_dataset/TFT_Challenger_MatchData.csv')

df_Champion_info = pd.read_csv('../../tft_game_dataset/TFT_Champion_CurrentVersion.csv')
df_Items_info = pd.read_csv('../../tft_game_dataset/TFT_Item_CurrentVersion.csv')


In [33]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['user_tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.shape

(399998, 9)

---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [35]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {df_all_match.shape}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: (399930, 9)


### 3-2. 컬럼명 소문자로 통일

In [36]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'user_tier'],
      dtype='str')


### 3-3. 시즌2 데이터 삭제

In [37]:
df_all_match_2 = df_all_match.copy()

# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_1 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_1 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_1)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_1)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396698, 10)


### 3-4. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [38]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65988
Name: count, dtype: int64

In [39]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [40]:
# TemplateTrait 키가 있는 행 수 확인 → (0,10)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 10)

### 3-5. ranked=0인 데이터 삭제
- game_id 단위로 진행

In [41]:
# ranked==0인 행의 gameId 추출
drop_game_ids_2 = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_2)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (396654, 10)


### 3-6. USER_ID 컬럼 제작
- `KR-USER-(인덱스+1)`의 규칙으로 부여

In [42]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

df_all_match_2['user_id'] # (396654, 11)

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396654, dtype: str

In [44]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

df_all_match_2['player_cnt'].value_counts()

player_cnt
8     396336
16       304
7         14
Name: count, dtype: int64

---
---

### **챔피언과 콤비네이션이 모두 {}인 행 삭제**

In [ ]:
# 두 컬럼 모두 빈 딕셔너리인 행 삭제
drop_idx_3 = df_all_match_2[
    df_all_match_2['combination'].apply(lambda x: ast.literal_eval(x) == {}) &
    df_all_match_2['champion'].apply(lambda x: ast.literal_eval(x) == {})
].index

df_all_match_2 = df_all_match_2.drop(index=drop_idx_3) # (396391, 11)

In [ ]:
df_all_match_2.shape

### **combination 정보는 있지만 champion 정보는 없는 행 처리**

In [ ]:
# flag_1 컬럼 생성
# combination은 값이 있고 champion은 빈 딕셔너리인 경우 True -> 1

df_all_match_2['flag_1'] = (
    df_all_match_2['combination'].apply(lambda x: ast.literal_eval(x) != {}) &
    df_all_match_2['champion'].apply(lambda x: ast.literal_eval(x) == {})
).astype(int)

In [ ]:
df_all_match_2['flag_1'].value_counts()

### **champion 정보는 있지만 combination 정보는 없는 행 처리**

In [ ]:
# flag_2 컬럼 생성
# combination은 값이 있고 champion은 빈 딕셔너리인 경우 True -> 1

df_all_match_2['flag_2'] = (
    df_all_match_2['combination'].apply(lambda x: ast.literal_eval(x) == {}) &
    df_all_match_2['champion'].apply(lambda x: ast.literal_eval(x) != {})
).astype(int)

# 결과 확인
df_all_match_2['flag_2'].value_counts()

---

In [ ]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['user_tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

print(len((mixed_gameids)))

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['user_tier'].value_counts()

In [ ]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

In [ ]:
# 기본값 normal로 설정
df_all_match_2['mix_flag'] = 'normal'

# gameId별 최고/최저 티어 추출
highest_tier = {}
lowest_tier = {}

for gameid, group in df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['user_tier']:
    tier_numbers = [tier_order[t] for t in group]
    
    max_tier_number = max(tier_numbers)
    min_tier_number = min(tier_numbers)
    
    for t in tier_order:
        if tier_order[t] == max_tier_number:
            highest_tier[gameid] = t
        if tier_order[t] == min_tier_number:
            lowest_tier[gameid] = t

# 믹스매치 행에 high/low 플래그 부여
for idx, row in df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].iterrows():
    if row['user_tier'] == highest_tier[row['gameid']]:
        df_all_match_2.loc[idx, 'mix_flag'] = 'high'
    else:
        df_all_match_2.loc[idx, 'mix_flag'] = 'low'

# 확인
df_all_match_2['mix_flag'].value_counts()